# Module 1 
4 I's of oppression

In [1]:
# performing imports
# python libraries
import os
import string
import re
from IPython.display import display, HTML
import warnings
import tqdm
from typing import List, Tuple, Dict

# Basic ML libraries and scikit learn libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from matplotlib.colors import ListedColormap

# LDA libraries
import nltk
import gensim.corpora as corpora
import gensim
import gensim.models
from gensim.models import CoherenceModel
from gensim .models import TfidfModel
import spacy
import pyLDAvis
import pyLDAvis.gensim_models

# Bert and Pytorch libraries
import torch
import accelerate
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize
from torch.utils.data import DataLoader
from datasets import Dataset, Features, Value, Sequence
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, BertModel, AdamW

In [2]:
# Getting current working directory
cwd = os.getcwd()
cwd

'D:\\My Work\\Computer Science Courses and Projects\\EMP_Work'

## Module 1

### 1.Data Preparation

In [3]:
# Reading CSV for module 1
M1 = pd.read_csv("Module 1.csv")

In [4]:
display(M1)

,ID,Campus,Comment
0,32,Bakersfield,NaN
1,36,Chico,Something that I noticed is how I feel about m...
2,128,Chico,"As an intersectional scholar, I had experience..."
3,25,Dominguez Hills,It was great being able to reflect on the iden...
4,91,Dominguez Hills,I noticed that it was easy for me to identify ...
5,186,Dominguez Hills,I also identify with the geographic aspect. Wh...
6,11,East Bay,A highlight from my experience was considering...
7,81,East Bay,I’m 42 years old. It was hard to think about w...
8,201,East Bay,I've worked to ensure I am aware of my own tra...
9,35,Fresno,I enjoy having time to engage in activities li...


In [5]:
# removing all the Nan values from the dataframe
print(M1.isna())

       ID  Campus  Comment
0   False   False     True
1   False   False    False
2   False   False    False
3   False   False    False
4   False   False    False
5   False   False    False
6   False   False    False
7   False   False    False
8   False   False    False
9   False   False    False
10  False   False    False
11  False   False    False
12  False   False    False
13  False   False    False
14  False   False    False
15  False   False    False
16  False   False    False
17  False   False    False
18  False   False    False
19  False   False    False
20  False   False    False


In [6]:
M1 = M1.dropna()
M1 = M1.reset_index(drop=True)

In [7]:
# checking the data types for the dataframe
M1.describe()

,ID
count,20.000000
mean,76.000000
std,67.734777
min,6.000000
25%,20.250000
50%,58.500000
75%,104.750000
max,212.000000


In [8]:
M1.dtypes

ID          int64
Campus     object
Comment    object
dtype: object

In [9]:
# converting the second and third column into string datatype for further processing
M1['Campus']=M1['Campus'].astype(pd.StringDtype())
M1['Comment']=M1['Comment'].astype(pd.StringDtype())
M1.dtypes

ID                  int64
Campus     string[python]
Comment    string[python]
dtype: object

### 2.Text preprocessing

In [10]:
# Text preparation
# removing punctuations ("!”#$%&'()*+,-./:;?@[\]^_`{|}~")
def remove_punctuation(text):
    punctuationfree="".join([i for i in text if i not in string.punctuation])
    return punctuationfree
M1['Clean_Comment'] = M1['Comment'].apply(lambda x:remove_punctuation(x))
M1.head()

,ID,Campus,Comment,Clean_Comment
0,36,Chico,Something that I noticed is how I feel about m...,Something that I noticed is how I feel about m...
1,128,Chico,"As an intersectional scholar, I had experience...",As an intersectional scholar I had experience ...
2,25,Dominguez Hills,It was great being able to reflect on the iden...,It was great being able to reflect on the iden...
3,91,Dominguez Hills,I noticed that it was easy for me to identify ...,I noticed that it was easy for me to identify ...
4,186,Dominguez Hills,I also identify with the geographic aspect. Wh...,I also identify with the geographic aspect Whe...


In [11]:
# lower casing the text
M1['Clean_Comment'] = M1['Clean_Comment'].apply(lambda x:x.lower())

In [12]:
display(M1)

,ID,Campus,Comment,Clean_Comment
0,36,Chico,Something that I noticed is how I feel about m...,something that i noticed is how i feel about m...
1,128,Chico,"As an intersectional scholar, I had experience...",as an intersectional scholar i had experience ...
2,25,Dominguez Hills,It was great being able to reflect on the iden...,it was great being able to reflect on the iden...
3,91,Dominguez Hills,I noticed that it was easy for me to identify ...,i noticed that it was easy for me to identify ...
4,186,Dominguez Hills,I also identify with the geographic aspect. Wh...,i also identify with the geographic aspect whe...
5,11,East Bay,A highlight from my experience was considering...,a highlight from my experience was considering...
6,81,East Bay,I’m 42 years old. It was hard to think about w...,i’m 42 years old it was hard to think about wh...
7,201,East Bay,I've worked to ensure I am aware of my own tra...,ive worked to ensure i am aware of my own trai...
8,35,Fresno,I enjoy having time to engage in activities li...,i enjoy having time to engage in activities li...
9,15,Long Beach,This was an interesting exercise. I am the onl...,this was an interesting exercise i am the only...


In [13]:
# Tokenization
def tokenization(text):
    tokens = re.split(r'\W+',text)
    return tokens
M1['Token_Comment'] = M1['Clean_Comment'].apply(lambda x:tokenization(x))
M1

,ID,Campus,Comment,Clean_Comment,Token_Comment
0,36,Chico,Something that I noticed is how I feel about m...,something that i noticed is how i feel about m...,"[something, that, i, noticed, is, how, i, feel..."
1,128,Chico,"As an intersectional scholar, I had experience...",as an intersectional scholar i had experience ...,"[as, an, intersectional, scholar, i, had, expe..."
2,25,Dominguez Hills,It was great being able to reflect on the iden...,it was great being able to reflect on the iden...,"[it, was, great, being, able, to, reflect, on,..."
3,91,Dominguez Hills,I noticed that it was easy for me to identify ...,i noticed that it was easy for me to identify ...,"[i, noticed, that, it, was, easy, for, me, to,..."
4,186,Dominguez Hills,I also identify with the geographic aspect. Wh...,i also identify with the geographic aspect whe...,"[i, also, identify, with, the, geographic, asp..."
5,11,East Bay,A highlight from my experience was considering...,a highlight from my experience was considering...,"[a, highlight, from, my, experience, was, cons..."
6,81,East Bay,I’m 42 years old. It was hard to think about w...,i’m 42 years old it was hard to think about wh...,"[i, m, 42, years, old, it, was, hard, to, thin..."
7,201,East Bay,I've worked to ensure I am aware of my own tra...,ive worked to ensure i am aware of my own trai...,"[ive, worked, to, ensure, i, am, aware, of, my..."
8,35,Fresno,I enjoy having time to engage in activities li...,i enjoy having time to engage in activities li...,"[i, enjoy, having, time, to, engage, in, activ..."
9,15,Long Beach,This was an interesting exercise. I am the onl...,this was an interesting exercise i am the only...,"[this, was, an, interesting, exercise, i, am, ..."


In [14]:
# Also removing the stopwords
# using predefined nltk library stop words
stopwords = nltk.corpus.stopwords.words('english')
stopwords[0:10]

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're"]

In [15]:
#defining the function to remove stopwords from tokenized text
def remove_stopwords(text):
    output= [i for i in text if i not in stopwords]
    return output
M1['Token_no_stopwords'] = M1['Token_Comment'].apply(lambda x : remove_stopwords(x))
M1

,ID,Campus,Comment,Clean_Comment,Token_Comment,Token_no_stopwords
0,36,Chico,Something that I noticed is how I feel about m...,something that i noticed is how i feel about m...,"[something, that, i, noticed, is, how, i, feel...","[something, noticed, feel, labels, versus, wou..."
1,128,Chico,"As an intersectional scholar, I had experience...",as an intersectional scholar i had experience ...,"[as, an, intersectional, scholar, i, had, expe...","[intersectional, scholar, experience, discussi..."
2,25,Dominguez Hills,It was great being able to reflect on the iden...,it was great being able to reflect on the iden...,"[it, was, great, being, able, to, reflect, on,...","[great, able, reflect, identity, map, brought,..."
3,91,Dominguez Hills,I noticed that it was easy for me to identify ...,i noticed that it was easy for me to identify ...,"[i, noticed, that, it, was, easy, for, me, to,...","[noticed, easy, identify, elements, better, pu..."
4,186,Dominguez Hills,I also identify with the geographic aspect. Wh...,i also identify with the geographic aspect whe...,"[i, also, identify, with, the, geographic, asp...","[also, identify, geographic, aspect, return, r..."
5,11,East Bay,A highlight from my experience was considering...,a highlight from my experience was considering...,"[a, highlight, from, my, experience, was, cons...","[highlight, experience, considering, places, g..."
6,81,East Bay,I’m 42 years old. It was hard to think about w...,i’m 42 years old it was hard to think about wh...,"[i, m, 42, years, old, it, was, hard, to, thin...","[42, years, old, hard, think, put, identities,..."
7,201,East Bay,I've worked to ensure I am aware of my own tra...,ive worked to ensure i am aware of my own trai...,"[ive, worked, to, ensure, i, am, aware, of, my...","[ive, worked, ensure, aware, traits, exercise,..."
8,35,Fresno,I enjoy having time to engage in activities li...,i enjoy having time to engage in activities li...,"[i, enjoy, having, time, to, engage, in, activ...","[enjoy, time, engage, activities, like, forces..."
9,15,Long Beach,This was an interesting exercise. I am the onl...,this was an interesting exercise i am the only...,"[this, was, an, interesting, exercise, i, am, ...","[interesting, exercise, daughter, family, two,..."


In [16]:
# Adding a lemmatized column
wordnet_lemmatizer = WordNetLemmatizer()
def lemmatizer(text):
    lemm_text = [wordnet_lemmatizer.lemmatize(word) for word in text]
    return lemm_text
M1['Token_no_stopwords_lemmatized']=M1['Token_no_stopwords'].apply(lambda x:lemmatizer(x))

In [17]:
M1

,ID,Campus,Comment,Clean_Comment,Token_Comment,Token_no_stopwords,Token_no_stopwords_lemmatized
0,36,Chico,Something that I noticed is how I feel about m...,something that i noticed is how i feel about m...,"[something, that, i, noticed, is, how, i, feel...","[something, noticed, feel, labels, versus, wou...","[something, noticed, feel, label, versus, woul..."
1,128,Chico,"As an intersectional scholar, I had experience...",as an intersectional scholar i had experience ...,"[as, an, intersectional, scholar, i, had, expe...","[intersectional, scholar, experience, discussi...","[intersectional, scholar, experience, discussi..."
2,25,Dominguez Hills,It was great being able to reflect on the iden...,it was great being able to reflect on the iden...,"[it, was, great, being, able, to, reflect, on,...","[great, able, reflect, identity, map, brought,...","[great, able, reflect, identity, map, brought,..."
3,91,Dominguez Hills,I noticed that it was easy for me to identify ...,i noticed that it was easy for me to identify ...,"[i, noticed, that, it, was, easy, for, me, to,...","[noticed, easy, identify, elements, better, pu...","[noticed, easy, identify, element, better, pur..."
4,186,Dominguez Hills,I also identify with the geographic aspect. Wh...,i also identify with the geographic aspect whe...,"[i, also, identify, with, the, geographic, asp...","[also, identify, geographic, aspect, return, r...","[also, identify, geographic, aspect, return, r..."
5,11,East Bay,A highlight from my experience was considering...,a highlight from my experience was considering...,"[a, highlight, from, my, experience, was, cons...","[highlight, experience, considering, places, g...","[highlight, experience, considering, place, gr..."
6,81,East Bay,I’m 42 years old. It was hard to think about w...,i’m 42 years old it was hard to think about wh...,"[i, m, 42, years, old, it, was, hard, to, thin...","[42, years, old, hard, think, put, identities,...","[42, year, old, hard, think, put, identity, wi..."
7,201,East Bay,I've worked to ensure I am aware of my own tra...,ive worked to ensure i am aware of my own trai...,"[ive, worked, to, ensure, i, am, aware, of, my...","[ive, worked, ensure, aware, traits, exercise,...","[ive, worked, ensure, aware, trait, exercise, ..."
8,35,Fresno,I enjoy having time to engage in activities li...,i enjoy having time to engage in activities li...,"[i, enjoy, having, time, to, engage, in, activ...","[enjoy, time, engage, activities, like, forces...","[enjoy, time, engage, activity, like, force, g..."
9,15,Long Beach,This was an interesting exercise. I am the onl...,this was an interesting exercise i am the only...,"[this, was, an, interesting, exercise, i, am, ...","[interesting, exercise, daughter, family, two,...","[interesting, exercise, daughter, family, two,..."


### 3.Text Representation

### 4.Modelling, Fine tuning, Improvements and Analysis

#### Approach 1: LDA

In [ ]:
# Converting the Tokens with no stopword column to a dictionary of words along with their frequency, adding a new column
id2word = corpora.Dictionary(M1['Token_no_stopwords'])
id2word_lemm = corpora.Dictionary(M1['Token_no_stopwords_lemmatized'])
corpus = []
corpus_lemm = []
for text in M1['Token_no_stopwords']:
    new = id2word.doc2bow(text)
    corpus.append(new)
for text in M1['Token_no_stopwords_lemmatized']:
    new_lemm = id2word_lemm.doc2bow(text)
    corpus_lemm.append(new_lemm)
for i in range(20):
    # printing the final tuple
    print(corpus[i])
    #and assigning the tuple as a column in the main data frame
    M1['Final_tuple'] = corpus
    M1['Final_tuple_lemmatized'] = corpus_lemm

In [ ]:
# having a look at updated data frame
M1

In [ ]:
# 1st iteration for the LDA model training
lda_model_1 = gensim.models.ldamodel.LdaModel(corpus = corpus,
                                            id2word = id2word,
                                            num_topics = 4,
                                            random_state = 100,
                                            update_every = 1,
                                            chunksize = 1000,
                                            passes = 100,
                                            alpha = "auto")


In [ ]:
# analyzing the results
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model_1, corpus, id2word, mds = "mmds", R=4)
vis

In [ ]:
# LDA model training with lemmatized data
lda_model_2 = gensim.models.ldamodel.LdaModel(corpus = corpus_lemm,
                                            id2word = id2word_lemm,
                                            num_topics = 4,
                                            random_state = 100,
                                            update_every = 1,
                                            chunksize = 100,
                                            passes = 10,
                                            alpha = "auto")

In [ ]:
# analyzing the results
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model_2, corpus_lemm, id2word_lemm, mds = "mmds", R=4)
vis

In [ ]:
# Improving the results a bit more
# creation of bigrams and trigrams, so that the model recognizes a collection of words (bigram for 2 words that mean something together and trigram for three words that mean something together)
bigram_phrases = gensim.models.Phrases(M1['Token_no_stopwords_lemmatized'], min_count=5, threshold=50)
trigram_phrases = gensim.models.Phrases(bigram_phrases[M1['Token_no_stopwords_lemmatized']], threshold=50)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return [bigram[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram[bigram[doc]] for doc in texts]


data_bi = make_bigrams(M1['Token_no_stopwords_lemmatized'])
data_bi_tri = make_trigrams(data_bi)

print(data_bi_tri)

M1['Token_no_stopwords_lemmatized_bi_tri'] = data_bi_tri

# TF-IDF removal: Removal of most words that mean nothing or have very little importance in the overall context
id2word = corpora.Dictionary(data_bi_tri)

texts = data_bi_tri

corpus = [id2word.doc2bow(text) for text in texts]

tfidf = TfidfModel(corpus, id2word = id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range(0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] #reinitialize to be safe. You can skip this.
    tfidf_ids = [id for id, value in tfidf[bow]]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # The words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]  

#reassign        
corpus[i] = new_bow

In [ ]:
M1

In [ ]:
# Improved iteration for the LDA model training
lda_model_3 = gensim.models.ldamodel.LdaModel(corpus = corpus,
                                            id2word = id2word,
                                            num_topics = 4,
                                            random_state = 100,
                                            update_every = 1,
                                            chunksize = 1000,
                                            passes = 100,
                                            alpha = "auto")

In [ ]:
# Improved model visualization
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model_3, corpus, id2word, mds = "mmds", R=4)
vis

### 5.Deployment (Embedding the results topicwise into the DF)

In [ ]:
# Function to infer topics for a given preprocessed text
def infer_topics(text):
    bow = id2word.doc2bow(text)
    topics = lda_model_2[bow] # change the model name
    return topics

# Apply the function to each preprocessed text in the DataFrame
M1['topics'] = M1['Token_no_stopwords_lemmatized'].apply(infer_topics) # change the column wrt the model we are using

# Extract top words for each topic
num_words = 10  # Number of top words to consider for highlighting
topic_keywords = {topic_id: [word for word, _ in lda_model_2.show_topic(topic_id, topn=num_words)]
                  for topic_id in range(lda_model_2.num_topics)} # again change the model name here

# Define a color for each topic
topic_colors = {
    0: '#FF0000',  # Red
    1: '#00FF00',  # Green
    2: '#0000FF',  # Blue
    3: '#FFFF00',  # Yellow
    # Add more colors if you have more topics
}

# Function to highlight text with different colors for each topic
def highlight_text(text, keywords_colors):
    # Sort keywords by length to avoid partial overlaps (longer keywords first)
    sorted_keywords = sorted(keywords_colors.keys(), key=len, reverse=True)
    highlighted_text = text
    for keyword in sorted_keywords:
        color = keywords_colors[keyword]
        # Use non-capturing groups to prevent overlapping issues
        highlighted_text = re.sub(f'(?<!\w){re.escape(keyword)}(?!\w)', f'<span style="background-color:{color}">{keyword}</span>', highlighted_text, flags=re.IGNORECASE)
    return highlighted_text

# Apply highlighting to each response based on its topics
def highlight_response(row):
    highlighted_response = row['Comment']
    
    # Create a dictionary of keywords and their colors
    keywords_colors = {}
    for topic, _ in row['topics']:
        keywords = topic_keywords[topic]
        color = topic_colors.get(topic, '#FFFFFF')  # Default to white if color not found
        for keyword in keywords:
            keywords_colors[keyword] = color
    
    # Highlight text with all keywords
    highlighted_response = highlight_text(highlighted_response, keywords_colors)
    
    return highlighted_response

M1['highlighted_response'] = M1.apply(highlight_response, axis=1)

# Display the highlighted responses using IPython.display
for idx, row in M1.iterrows():
    display(HTML(f"<div><strong>Response {idx + 1}:</strong><br>{row['highlighted_response']}</div>"))

##### Key observations from the 1st approach
1. The LDA model seems to focus more on words rather than the context
2. able to take out different topics from one response, but for some responses the model seems to be oversaturated, i.e. it only focuses on one topic per response. It might be due to less data.
3. We are not able to separate/segregate the whole text into types of oppression but only a few words.
4. Also, due to the unsupervised nature, we cannot get the topic that the classification refers to despite complete classification, as the model doesn't consider the topic's meaning while classifying.

Therefore, it seems LDA, no matter how good we train will not be good enough W.R.T. requirement and might be time costly too. Therefore, in 2nd approach, we are gonna go with LLMs.

#### Approach 2 A: Sentence BERT (unsupervised)
Unsupervised: Find topics with just with pretrained BERT model

Supervised: train and fine tune Bert model with augmented data from AI on 4 types of oppression

In [ ]:
# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Generate BERT embeddings for each sentence in a response
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

# Tokenize each response into sentences and get embeddings for each sentence
def process_response(response):
    sentences = sent_tokenize(response)
    embeddings = np.array([get_bert_embeddings(sentence) for sentence in sentences])
    return sentences, embeddings

M1['processed_bert_1'] = M1['Comment'].apply(process_response)

# Function to cluster sentences within each response
def cluster_sentences(embeddings, max_clusters=4):
    num_sentences = embeddings.shape[0]
    num_clusters = min(max_clusters, num_sentences)
    
    if num_clusters < 2:
        # Assign a single cluster if not enough sentences
        return np.zeros(num_sentences, dtype=int)
    
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    kmeans.fit(embeddings)
    return kmeans.labels_

# Apply clustering to each response
M1['clusters_bert_1'] = M1['processed_bert_1'].apply(lambda x: cluster_sentences(x[1]))

# Define a color for each cluster
cluster_colors = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00']

# Function to highlight text based on clusters
def highlight_response(sentences, clusters):
    highlighted_text = ""
    for sentence, cluster in zip(sentences, clusters):
        color = cluster_colors[cluster % len(cluster_colors)]
        highlighted_text += f'<span style="background-color:{color}">{sentence}</span> '
    return highlighted_text

# Apply highlighting to each response
M1['highlighted_response_bert_1'] = M1.apply(lambda row: highlight_response(row['processed_bert_1'][0], row['clusters_bert_1']), axis=1)

# Display the highlighted responses using IPython.display
for idx, row in M1.iterrows():
    display(HTML(f"<div><strong>Response {idx + 1}:</strong><br>{row['highlighted_response_bert_1']}</div>"))
HTML('''<script>
code_show_err=false; 
function code_toggle_err() {
 if (code_show_err){
 $('div.output_stderr').hide();
 } else {
 $('div.output_stderr').show();
 }
 code_show_err = !code_show_err
} 
$( document ).ready(code_toggle_err);
</script>
To toggle on/off output_stderr, click <a href="javascript:code_toggle_err()">here</a>.''')

In [ ]:
# Using already tokenized data, More word by word approach

# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Generate BERT embeddings for each tokenized sentence
def get_bert_embeddings(tokens):
    text = ' '.join(tokens)
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use CLS token pooling
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# Tokenize each response into sentences and get embeddings for each sentence
def process_response(tokens):
    embeddings = np.array([get_bert_embeddings(token_list) for token_list in tokens])
    return tokens, embeddings

M1['processed_bert_2'] = M1['Token_Comment'].apply(process_response)

# Function to cluster sentences within each response
def cluster_sentences(embeddings, max_clusters=4):
    num_sentences = embeddings.shape[0]
    num_clusters = min(max_clusters, num_sentences)
    
    if num_clusters < 2:
        # Assign a single cluster if not enough sentences
        return np.zeros(num_sentences, dtype=int)
    
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    kmeans.fit(embeddings)
    return kmeans.labels_

# Apply clustering to each response
M1['clusters_bert_2'] = M1['processed_bert_2'].apply(lambda x: cluster_sentences(x[1]))


# Define a color for each cluster
cluster_colors = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00']

# Function to highlight text based on clusters
def highlight_response(sentences, clusters):
    highlighted_text = ""
    for sentence, cluster in zip(sentences, clusters):
        color = cluster_colors[cluster % len(cluster_colors)]
        highlighted_text += f'<span style="background-color:{color}">{sentence}</span> '
    return highlighted_text

# Apply highlighting to each response
M1['highlighted_response_bert_2'] = M1.apply(lambda row: highlight_response(row['processed_bert_2'][0], row['clusters_bert_2']), axis=1)

# Display the highlighted responses using IPython.display
for idx, row in M1.iterrows():
    display(HTML(f"<div><strong>Response {idx + 1}:</strong><br>{row['highlighted_response_bert_2']}</div>"))


##### key observations
1. The model clearly classifies the responses into 4 categories but, the topic are not given to the model. Thus, classification is not taking the topics and its meaning into account

Therefore, using a supervised approach will be a better option

In [ ]:
# more improvements

#### Approach 2 B: Supervised BERT model

For supervised BERT model training we will be taking data for the training from ChatGPT, on prompting it to generate some random responses from faculty on 4 types of oppression

In [18]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training data generated via AI, prompt: "Create 4 1st person responses from faculties of Fresno State when asked about the identity reflection. Include experiences from them or their students. Also, create responses that falls in each one of 4 I's of oppression"
responses_df = pd.DataFrame({
    'I of oppression': [
        'Idealogical', 'Idealogical', 'Idealogical',
        'Institutionalized', 'Institutionalized', 'Institutionalized',
        'Internalized', 'Internalized', 
        'Interpersonal', 'Interpersonal'
    ],
    'responses': [
        "As a faculty member with a strong belief in the efficacy of renewable energy, I often emphasize the importance of sustainable practices in my environmental science courses. While I strive to present various perspectives, my own ideological commitment to combating climate change can sometimes lead me to highlight information that supports my views more strongly than alternative viewpoints.",
        "My political ideology shapes how I approach discussions on governance and public policy in my political science classes. Despite my efforts to remain objective, my personal beliefs about the role of government often influence the examples and case studies I use, potentially skewing students' understanding of different political perspectives.",
        "In my sociology courses, my ideological perspective on social justice frequently influences how I present issues of inequality and systemic discrimination. While I aim to provide a balanced view, my strong commitment to social equity sometimes leads me to focus more on narratives that align with my values, which might affect students' engagement with diverse viewpoints.",
        "Institutional policies on research funding have sometimes limited my ability to pursue studies on topics I am passionate about. The focus on securing grants for high-priority research areas often means I must align my projects with institutional interests, which can restrict the scope of my academic exploration and reduce the diversity of research topics I can investigate.",
        "The institutional framework for curriculum development often constrains the flexibility I have in designing my courses. While I strive to incorporate innovative teaching methods and up-to-date content, I am frequently required to adhere to prescribed course outlines and standards set by the institution, which can sometimes hinder my ability to fully address emerging trends or diverse student needs.",
        "Institutional guidelines on student evaluations and grading can sometimes be at odds with my own educational philosophy. While I aim to provide fair and constructive feedback, I am often required to follow standardized grading criteria that may not fully capture the nuances of student performance or the holistic nature of their learning experiences.",
        "I occasionally struggle with feelings of inadequacy regarding my ability to address the diverse needs of my students. Despite my experience, I sometimes doubt whether I am effectively meeting the needs of all students, particularly those from underrepresented backgrounds. These internalized concerns can impact my confidence and effectiveness as a teacher.",
        "My self-doubt about my effectiveness as an advisor to students from marginalized communities sometimes leads me to second-guess my actions and decisions. Although I strive to be supportive and inclusive, my internalized fears about not being sufficiently understanding or helpful can hinder my ability to fully engage with and support these students.",
        "I frequently question whether my teaching practices are truly inclusive and supportive of students who face educational barriers similar to those I experienced. This internalized concern about not doing enough can lead to heightened self-scrutiny and a constant drive to improve, sometimes to the point of overextending myself.",
        "Interactions with colleagues can sometimes reveal underlying biases or power dynamics that influence our professional relationships. For example, I have noticed moments where subtle differences in how we communicate and collaborate reflect broader societal inequalities. These interpersonal dynamics can affect the collaborative spirit within our department and influence our collective productivity."
    ]
})

responses_df

,I of oppression,responses
0,Idealogical,As a faculty member with a strong belief in th...
1,Idealogical,My political ideology shapes how I approach di...
2,Idealogical,"In my sociology courses, my ideological perspe..."
3,Institutionalized,Institutional policies on research funding hav...
4,Institutionalized,The institutional framework for curriculum dev...
5,Institutionalized,Institutional guidelines on student evaluation...
6,Internalized,I occasionally struggle with feelings of inade...
7,Internalized,My self-doubt about my effectiveness as an adv...
8,Interpersonal,I frequently question whether my teaching prac...
9,Interpersonal,Interactions with colleagues can sometimes rev...
